# AI Job Agent — PHASE 8: Agent (User ↔ Tools ↔ Jobs)

**الهدف:** واجهة محادثة تسمح للمستخدم يسأل بلغة طبيعية عن نتائج المطابقة
الجاهزة من Phase 5/6/7، والـ Agent يستدعي أدوات Python محلية (لا إعادة حساب)
للإجابة — بنفس روح صرامة Phase 7: **لا اختلاق نتائج، كل رقم أو مهارة
مذكورة لازم تكون موجودة فعليًا في أحد الملفات المحمّلة.**

```
phase7_output.json (Phase 7 — تفسيرات أعلى N وظيفة)
reranked_jobs.parquet (Phase 6 — تجمّع أوسع، اختياري)      ┐
matched_jobs.parquet (Phase 5 — كل الوظائف المؤهلة، اختياري) ┘  أدوات بحث/تصفية أوسع
         │
         ▼
    User message ──► Agent (gpt-5.6-luna + tool-calling) ──► استدعاء أداة Python محلية
         ▲                                                          │
         └──────────────────── رد نصي مبني فقط على مخرج الأداة ◄────┘
```

**نطاق هذا النوتبوك:** Agent استكشافي على نتائج جاهزة — **لا يعيد تشغيل**
Phase 3-6 (فيها استدعاءات embedding ونموذج cross-encoder ثقيل). لو المستخدم
غيّر تفضيلاته جذريًا (مدينة جديدة، راتب مختلف)، الحل الصحيح إعادة تشغيل
Phase 2 فما فوق، مو تعديل هذا الـ Agent.

**مصادر البيانات الثلاثة، بترتيب التفصيل:**

| الملف | من مرحلة | ماذا يوفر | إلزامي؟ |
|---|---|---|---|
| `phase7_output.json` | 7 | `why_match`, مهارات/مؤهلات/تعليم مطابقة وناقصة, `learning_areas` | ✅ مطلوب |
| `reranked_jobs.parquet` | 6 | تجمّع أوسع (حتى 50)، بدون نص تفسيري | اختياري |
| `matched_jobs.parquet` | 5 | كل الوظائف اللي عدّت Phase 4 (city/country/work_arrangement...) | اختياري |

كل ملف اختياري يوسّع قدرة الـ Agent (تصفية حسب المدينة، تصفح أبعد من أعلى N)
لكن الـ Agent يشتغل بـ`phase7_output.json` وحده لو ما رفعت الباقي.

---
## 1. Setup

In [ ]:
!pip install -q openai

In [ ]:
import json
import re
import shutil
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 100)

WORK = Path("work")
OUT = Path("outputs") / "live_candidate"
for d in (WORK, OUT):
    d.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")
print("RUN_ID:", RUN_ID)

In [ ]:
CONFIG = {
    "agent_model": "gpt-5.6-luna",   # نفس الموديل المستخدم في Phase 7
    "max_tool_iterations": 6,        # حد أقصى لعدد استدعاءات الأدوات ضمن رد واحد (يمنع حلقة لا منتهية)
}

from openai import OpenAI

try:
    from google.colab import userdata
    api_key = userdata.get("openai_key_new")
except Exception:
    import os
    api_key = os.environ.get("OPENAI_API_KEY")

assert api_key, "OpenAI API key not found"
client = OpenAI(api_key=api_key)

print("Client ready. Model:", CONFIG["agent_model"])

---
## 2. Load Phase 5/6/7 outputs

In [ ]:
from google.colab import files as _colab_files

print("ارفع phase7_output.json (إلزامي)، وإن رغبت: "
      "reranked_jobs.parquet + matched_jobs.parquet (اختياري):")
uploaded = _colab_files.upload()

DATA_DIR = WORK / "inputs"
DATA_DIR.mkdir(exist_ok=True)
for name, content in uploaded.items():
    (DATA_DIR / name).write_bytes(content)

print("\nSaved:", [f.name for f in DATA_DIR.iterdir()])

In [ ]:
p7_path = DATA_DIR / "phase7_output.json"
assert p7_path.exists(), (
    "phase7_output.json غير موجود. هذا الملف إلزامي — هو المصدر الوحيد "
    "للتفسيرات النصية (why_match) واقتراحات التحسين."
)

phase7 = json.loads(p7_path.read_text(encoding="utf-8"))
assert "jobs" in phase7 and isinstance(phase7["jobs"], list)

CANDIDATE_ID = phase7.get("candidate_id")
JOBS_BY_ID = {job["job_id"]: job for job in phase7["jobs"]}

print(f"Candidate    : {CANDIDATE_ID}")
print(f"Phase 7 jobs : {len(JOBS_BY_ID)} (with full explanation)")

p6_path = DATA_DIR / "reranked_jobs.parquet"
p5_path = DATA_DIR / "matched_jobs.parquet"

reranked_df = pd.read_parquet(p6_path) if p6_path.exists() else None
matched_df = pd.read_parquet(p5_path) if p5_path.exists() else None

print(f"Phase 6 pool : {len(reranked_df) if reranked_df is not None else 'not provided'}")
print(f"Phase 5 pool : {len(matched_df) if matched_df is not None else 'not provided'}")

# أوسع مجموعة متاحة للتصفح/التصفية (تفضّل Phase 5 لأنها الأشمل بالأعمدة)
BROWSE_POOL = matched_df if matched_df is not None else reranked_df
if BROWSE_POOL is not None:
    print(f"Browse pool  : {len(BROWSE_POOL)} jobs (from "
          f"{'Phase 5' if matched_df is not None else 'Phase 6'})")
else:
    print("Browse pool  : none — tools restricted to the "
          f"{len(JOBS_BY_ID)} jobs in phase7_output.json only.")

---
## 3. Tools — pure Python, no LLM involved

هذي الدوال **لا تستدعي أي نموذج**. تقرأ فقط من `JOBS_BY_ID` و`BROWSE_POOL`
المُحمَّلة أعلاه. هذا يضمن إن أي رقم أو مهارة يذكرها الـ Agent لاحقًا
موجودة فعليًا في ملفات Phase 5/6/7 — لا اختلاق ممكن على مستوى الأداة نفسها.

In [ ]:
def _job_not_found(job_id: str) -> dict:
    return {
        "error": "job_not_found",
        "job_id": job_id,
        "message": f"لا توجد وظيفة بهذا المعرّف ({job_id}) في النتائج المحمّلة.",
    }


def get_job_explanation(job_id: str) -> dict:
    """التفسير الكامل لوظيفة واحدة من Phase 7: لماذا تطابق، مهارات/مؤهلات/
    تعليم مطابق وناقص، واقتراحات التحسين. يعمل فقط للوظائف ضمن أعلى N
    (الموجودة في phase7_output.json)."""

    job = JOBS_BY_ID.get(job_id)
    if job is None:
        return _job_not_found(job_id)
    return job


def list_top_jobs(limit: int = 10) -> list:
    """أعلى الوظائف تطابقًا (من Phase 7، مرتبة حسب match_score تنازليًا)."""

    jobs = sorted(JOBS_BY_ID.values(), key=lambda j: j.get("match_score", 0), reverse=True)
    limit = max(1, min(limit, len(jobs)))
    return [
        {
            "job_id": j["job_id"],
            "title": j["title"],
            "company": j["company"],
            "city": (j.get("location") or {}).get("city"),
            "match_score": j["match_score"],
        }
        for j in jobs[:limit]
    ]


def compare_jobs(job_id_a: str, job_id_b: str) -> dict:
    """مقارنة جنبًا لجنب بين وظيفتين — الدرجة، المهارات المطابقة والناقصة."""

    a, b = JOBS_BY_ID.get(job_id_a), JOBS_BY_ID.get(job_id_b)
    if a is None:
        return _job_not_found(job_id_a)
    if b is None:
        return _job_not_found(job_id_b)

    def brief(j):
        return {
            "job_id": j["job_id"], "title": j["title"], "company": j["company"],
            "match_score": j["match_score"],
            "n_matched_skills": len(j.get("matched_skills", [])),
            "n_missing_skills": len(j.get("missing_skills", [])),
            "missing_skills": j.get("missing_skills", []),
        }

    return {"job_a": brief(a), "job_b": brief(b)}


def skill_gap_summary() -> dict:
    """أكثر المهارات الناقصة تكرارًا عبر كل الوظائف في Phase 7 — يجيب على
    'وش أهم شيء أطوّره؟' بدون الحاجة تمر على كل وظيفة يدويًا."""

    from collections import Counter

    counter = Counter()
    for job in JOBS_BY_ID.values():
        counter.update(job.get("missing_skills", []))

    if not counter:
        return {"message": "لا توجد مهارات ناقصة مسجّلة — تطابق ممتاز عبر كل الوظائف.", "gaps": []}

    return {
        "gaps": [
            {"skill": skill, "missing_in_n_jobs": count}
            for skill, count in counter.most_common(10)
        ],
        "n_jobs_analyzed": len(JOBS_BY_ID),
    }


def get_learning_plan(job_id: str) -> dict:
    """اقتراحات التحسين (learning_areas) لوظيفة محددة من Phase 7."""

    job = JOBS_BY_ID.get(job_id)
    if job is None:
        return _job_not_found(job_id)
    return {
        "job_id": job_id,
        "title": job["title"],
        "learning_areas": job.get("learning_areas", []),
    }


def browse_jobs(city: str = None, country: str = None,
                 work_arrangement: str = None,
                 min_match_score: float = None,
                 limit: int = 15) -> dict:
    """تصفية/تصفح أوسع من أعلى N — يحتاج matched_jobs.parquet (Phase 5) أو
    reranked_jobs.parquet (Phase 6) مرفوعًا. يرجع أعمدة أساسية فقط
    (بدون why_match، غير متوفر خارج مجموعة Phase 7)."""

    if BROWSE_POOL is None:
        return {
            "error": "no_browse_pool",
            "message": "لم يتم رفع matched_jobs.parquet أو reranked_jobs.parquet — "
                       "هذه الأداة تحتاج أحدهما. استخدم list_top_jobs بدلًا منها "
                       "للاستعلام ضمن أعلى الوظائف المتاحة.",
        }

    df = BROWSE_POOL.copy()

    if city:
        df = df[df["city"].astype(str).str.contains(city, case=False, na=False)]
    if country:
        df = df[df["country"].astype(str).str.contains(country, case=False, na=False)]
    if work_arrangement and "work_arrangement" in df.columns:
        df = df[df["work_arrangement"].astype(str).str.lower() == work_arrangement.lower()]
    if min_match_score is not None:
        score_col = "match_score" if "match_score" in df.columns else "objective_rank_score"
        df = df[df[score_col] >= min_match_score]

    score_col = "match_score" if "match_score" in df.columns else "objective_rank_score"
    df = df.sort_values(score_col, ascending=False)

    cols = [c for c in ["job_id", "title_clean", "company_clean", "city", "country",
                         "work_arrangement", score_col] if c in df.columns]

    limit = max(1, min(limit, 50))
    results = df[cols].head(limit).to_dict(orient="records")

    return {"count": len(results), "total_matches": len(df), "results": results}


def get_candidate_summary() -> dict:
    """ملخص عام حول جلسة المطابقة — عدد الوظائف، متوسط الدرجة، أفضل تطابق."""

    if not JOBS_BY_ID:
        return {"message": "لا توجد وظائف محمّلة."}

    scores = [j["match_score"] for j in JOBS_BY_ID.values()]
    best = max(JOBS_BY_ID.values(), key=lambda j: j["match_score"])

    return {
        "candidate_id": CANDIDATE_ID,
        "n_jobs_evaluated": len(JOBS_BY_ID),
        "avg_match_score": round(float(np.mean(scores)), 2),
        "best_match": {"job_id": best["job_id"], "title": best["title"],
                        "company": best["company"], "match_score": best["match_score"]},
    }


print("Tool functions defined:")
for fn in [get_job_explanation, list_top_jobs, compare_jobs, skill_gap_summary,
           get_learning_plan, browse_jobs, get_candidate_summary]:
    print(" -", fn.__name__)

### 3.1 Sanity checks — نفس روح فحوصات Phase 5/6 قبل الوصل بالـ LLM

اختبر كل دالة يدويًا على البيانات المحمّلة فعليًا قبل ما تربطها بالـ Agent.
لو أي دالة ترجع خطأ هنا، المشكلة بالبيانات مو بالـ Agent — أصلحها الآن.

In [ ]:
print("=== list_top_jobs(3) ===")
print(json.dumps(list_top_jobs(3), indent=2, ensure_ascii=False))

print("\n=== get_candidate_summary() ===")
print(json.dumps(get_candidate_summary(), indent=2, ensure_ascii=False))

print("\n=== skill_gap_summary() ===")
print(json.dumps(skill_gap_summary(), indent=2, ensure_ascii=False))

_first_job_id = next(iter(JOBS_BY_ID))
print(f"\n=== get_job_explanation('{_first_job_id}') ===")
print(json.dumps(get_job_explanation(_first_job_id), indent=2, ensure_ascii=False)[:800])

print(f"\n=== get_job_explanation('nonexistent_id') — should error gracefully ===")
print(json.dumps(get_job_explanation("nonexistent_id"), indent=2, ensure_ascii=False))

if BROWSE_POOL is not None:
    print("\n=== browse_jobs(limit=3) ===")
    print(json.dumps(browse_jobs(limit=3), indent=2, ensure_ascii=False)[:600])
else:
    print("\n=== browse_jobs() — no pool uploaded, checking graceful error ===")
    print(json.dumps(browse_jobs(), indent=2, ensure_ascii=False))

---
## 4. Tool schemas (for the LLM's function-calling)

وصف رسمي لكل أداة بصيغة JSON Schema حتى يقدر الموديل يقرر متى ومتى يستدعيها.
الأسماء هنا يجب أن تطابق حرفيًا أسماء الدوال بالقسم 3.

In [ ]:
TOOLS = [
    {
        "type": "function",
        "name": "list_top_jobs",
        "description": "أعلى الوظائف تطابقًا مع المرشح، مرتبة حسب درجة التطابق. "
                        "استخدمها كبداية لأي سؤال عام مثل 'وش أفضل وظيفة لي؟'.",
        "parameters": {
            "type": "object",
            "properties": {
                "limit": {"type": "integer", "description": "عدد الوظائف المطلوبة", "default": 10},
            },
            "required": [],
        },
    },
    {
        "type": "function",
        "name": "get_job_explanation",
        "description": "التفسير الكامل لوظيفة محددة: لماذا تطابق، المهارات/المؤهلات/"
                        "التعليم المطابق والناقص، واقتراحات التحسين.",
        "parameters": {
            "type": "object",
            "properties": {
                "job_id": {"type": "string", "description": "معرّف الوظيفة"},
            },
            "required": ["job_id"],
        },
    },
    {
        "type": "function",
        "name": "compare_jobs",
        "description": "مقارنة جنبًا لجنب بين وظيفتين محددتين بمعرّفيهما.",
        "parameters": {
            "type": "object",
            "properties": {
                "job_id_a": {"type": "string"},
                "job_id_b": {"type": "string"},
            },
            "required": ["job_id_a", "job_id_b"],
        },
    },
    {
        "type": "function",
        "name": "skill_gap_summary",
        "description": "أكثر المهارات الناقصة تكرارًا عبر كل الوظائف المُطابَقة — "
                        "يجيب على أسئلة مثل 'وش أهم شيء أطوّره؟'.",
        "parameters": {"type": "object", "properties": {}, "required": []},
    },
    {
        "type": "function",
        "name": "get_learning_plan",
        "description": "اقتراحات التعلّم/التحسين المرتبطة بوظيفة محددة.",
        "parameters": {
            "type": "object",
            "properties": {"job_id": {"type": "string"}},
            "required": ["job_id"],
        },
    },
    {
        "type": "function",
        "name": "browse_jobs",
        "description": "تصفية/تصفح الوظائف حسب المدينة، الدولة، نمط العمل، أو حد أدنى "
                        "لدرجة التطابق. يغطي مجموعة أوسع من أعلى N. يرجع خطأ واضح "
                        "لو لم تُرفع بيانات Phase 5/6.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string"},
                "country": {"type": "string"},
                "work_arrangement": {"type": "string",
                                      "enum": ["On-site", "Remote", "Hybrid", "Not Specified"]},
                "min_match_score": {"type": "number"},
                "limit": {"type": "integer", "default": 15},
            },
            "required": [],
        },
    },
    {
        "type": "function",
        "name": "get_candidate_summary",
        "description": "ملخص عام لجلسة المطابقة: عدد الوظائف، متوسط الدرجة، أفضل تطابق.",
        "parameters": {"type": "object", "properties": {}, "required": []},
    },
]

TOOL_FUNCTIONS = {
    "list_top_jobs": list_top_jobs,
    "get_job_explanation": get_job_explanation,
    "compare_jobs": compare_jobs,
    "skill_gap_summary": skill_gap_summary,
    "get_learning_plan": get_learning_plan,
    "browse_jobs": browse_jobs,
    "get_candidate_summary": get_candidate_summary,
}

assert set(t["name"] for t in TOOLS) == set(TOOL_FUNCTIONS.keys()), \
    "TOOLS و TOOL_FUNCTIONS غير متطابقين — كل أداة معرّفة بالـ schema يجب أن يقابلها دالة"

print(f"{len(TOOLS)} tools registered:", list(TOOL_FUNCTIONS.keys()))

---
## 5. System prompt

نفس منطق الصرامة في Phase 7 — الـ Agent ممنوع يخترع أي حقيقة، ولازم يستخدم
أداة قبل أي ادعاء عن وظيفة محددة.

In [ ]:
SYSTEM_PROMPT = f"""
You are the conversational agent in an AI Job Matching system, talking
directly to the candidate (candidate_id: {CANDIDATE_ID}).

You have access to tools that query already-computed matching results from
Phase 5 (scoring), Phase 6 (re-ranking), and Phase 7 (LLM explanations).
You do NOT compute or estimate match scores yourself — you only report what
the tools return.

STRICT GROUNDING RULES

1. Never state a match score, skill, qualification, or education fact about
   a specific job without first calling a tool to retrieve it.
2. Never invent a job_id. Only reference job_ids returned by a tool.
3. If a tool returns an error (e.g. job_not_found, no_browse_pool), explain
   the limitation to the user in plain language — do not pretend the data
   exists.
4. If the user asks something outside your tools' scope (e.g. "change my
   desired salary", "search a different city not in the loaded pool"),
   explain that this requires re-running earlier phases of the pipeline,
   not something you can do here.
5. When listing multiple jobs, always include the job_id so the user can
   ask follow-up questions about a specific one.
6. Keep answers concise and in the user's language (Arabic or English,
   mirror what they use).
7. You may call more than one tool in sequence if a question needs it
   (e.g. "compare my top 2 jobs" → list_top_jobs then compare_jobs).

Be warm and helpful, like a career advisor — but every factual claim must
trace back to a tool call.
"""

print(f"System prompt: {len(SYSTEM_PROMPT)} chars")

---
## 6. Agent loop (function-calling)

تنفيذ حلقة استدعاء الأدوات القياسية عبر OpenAI Responses API: الموديل يقرر
يستدعي أداة → ننفّذها محليًا → نرجّع النتيجة له → يكرر لو احتاج → يرد نص نهائي.
`max_tool_iterations` يمنع أي حلقة لا منتهية نظريًا.

In [ ]:
def run_tool_call(name: str, arguments: dict) -> dict:
    """Execute a local tool function by name, never raising to the caller."""

    fn = TOOL_FUNCTIONS.get(name)
    if fn is None:
        return {"error": "unknown_tool", "message": f"No such tool: {name}"}
    try:
        return fn(**arguments)
    except Exception as exc:
        return {"error": "tool_execution_failed", "message": repr(exc)[:300]}


def ask_agent(user_message: str, history: list) -> tuple:
    """One conversational turn. `history` is the running list of Responses-API
    input items (persisted by the caller across turns). Returns
    (assistant_text, updated_history, tool_calls_made)."""

    history = history + [{"role": "user", "content": user_message}]
    tool_calls_made = []

    for _ in range(CONFIG["max_tool_iterations"]):
        response = client.responses.create(
            model=CONFIG["agent_model"],
            instructions=SYSTEM_PROMPT,
            input=history,
            tools=TOOLS,
        )

        function_calls = [item for item in response.output if item.type == "function_call"]

        if not function_calls:
            assistant_text = response.output_text
            history = history + [{"role": "assistant", "content": assistant_text}]
            return assistant_text, history, tool_calls_made

        # الموديل طلب أداة (أو أكثر) — نفّذها محليًا وأرجع النتيجة
        history = history + list(response.output)

        for call in function_calls:
            args = json.loads(call.arguments) if call.arguments else {}
            result = run_tool_call(call.name, args)
            tool_calls_made.append({"tool": call.name, "arguments": args})

            history = history + [{
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(result, ensure_ascii=False),
            }]

    # تجاوز الحد الأقصى للتكرارات — نادر، لكن نتعامل معه بوضوح بدل الانهيار
    fallback = ("عذرًا، احتجت عدد كبير من الاستعلامات للإجابة على سؤالك ولم "
                "أصل لنتيجة نهائية. جرّب صياغة أبسط أو أكثر تحديدًا.")
    history = history + [{"role": "assistant", "content": fallback}]
    return fallback, history, tool_calls_made


print("Agent loop ready.")

### 6.1 Grounding check (best-effort)

فحص إضافي بعد كل رد: هل أي `job_id`-مثل ذُكر بالرد فعليًا موجود بالبيانات؟
هذا لا يمنع الرد (الموديل قد يذكر معرّفًا بسياق توضيحي)، لكنه يُنبّهك لمراجعة
يدوية لو الـ Agent بدأ "يختلق" معرّفات وظائف.

In [ ]:
KNOWN_JOB_IDS = set(JOBS_BY_ID.keys())
if BROWSE_POOL is not None and "job_id" in BROWSE_POOL.columns:
    KNOWN_JOB_IDS |= set(BROWSE_POOL["job_id"].astype(str))

_JOB_ID_PATTERN = re.compile(r"\bjob_[a-zA-Z0-9_]{3,}\b")


def check_grounding(assistant_text: str) -> list:
    """Return any job-id-looking tokens mentioned in the reply that are not
    in the known set — a signal (not proof) of possible fabrication."""

    mentioned = set(_JOB_ID_PATTERN.findall(assistant_text))
    unknown = mentioned - KNOWN_JOB_IDS
    return sorted(unknown)


print(f"Known job_ids for grounding check: {len(KNOWN_JOB_IDS)}")

---
## 7. Chat

شغّل هذي الخلية وابدأ تسأل. `history` تُحافظ على السياق بين الأسئلة ضمن
نفس التشغيل — أعد تشغيل الخلية لو تبي تبدأ محادثة جديدة من الصفر.

In [ ]:
history = []
CONVERSATION_LOG = []


def chat(message: str) -> None:
    """اطبع رد الـ Agent، وسجّل الاستدعاءات والتحقق من الأسس تلقائيًا."""

    global history

    reply, history, calls = ask_agent(message, history)

    unknown_ids = check_grounding(reply)

    print(f"أنت: {message}")
    print(f"\nAgent: {reply}")

    if calls:
        print(f"\n[أدوات مُستدعاة: {', '.join(c['tool'] for c in calls)}]")
    if unknown_ids:
        print(f"\n[⚠️ تنبيه تحقق: ذُكرت معرّفات غير معروفة: {unknown_ids}]")

    CONVERSATION_LOG.append({
        "user": message, "assistant": reply,
        "tool_calls": calls, "unknown_ids_flagged": unknown_ids,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    })


print("جاهز. استخدم chat(\"سؤالك هنا\") لبدء المحادثة.")

### 7.1 أمثلة — جرّبها أو استبدلها بأسئلتك

In [ ]:
chat("وش أفضل وظيفة لي؟")

In [ ]:
chat("ليش هذي الوظيفة مناسبة لي بالضبط؟")

In [ ]:
chat("وش أهم شيء أحتاج أطوّره عشان أطابق أكثر؟")

In [ ]:
chat("قارن لي بين أفضل وظيفتين")

### 7.2 محادثة تفاعلية حرة (اختياري)

نافذة إدخال حيّة — اكتب `خروج` لإنهاء الجلسة.

In [ ]:
while True:
    user_input = input("أنت: ").strip()
    if user_input.lower() in ("خروج", "exit", "quit"):
        print("انتهت الجلسة.")
        break
    if not user_input:
        continue
    chat(user_input)
    print()

---
## 8. Save conversation log

In [ ]:
log_path = OUT / "agent_conversation_log.json"
log_path.write_text(
    json.dumps({
        "candidate_id": CANDIDATE_ID,
        "run_id": RUN_ID,
        "turns": CONVERSATION_LOG,
    }, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"Saved {log_path}  ({len(CONVERSATION_LOG)} turns)")

---
## 9. Data dictionary

In [ ]:
DATA_DICTIONARY = """# Phase 8 outputs — Data Dictionary

## live_candidate/agent_conversation_log.json

One record per conversational turn during this session.

| Field | Notes |
|---|---|
| user | The person's message. |
| assistant | The agent's final text reply (after any tool calls). |
| tool_calls | List of {tool, arguments} — every local function invoked to produce the reply. |
| unknown_ids_flagged | job_id-looking tokens in the reply not found in loaded data. Non-empty = review manually for possible fabrication. |
| timestamp | UTC ISO timestamp. |

## Tool contract (for extending this agent)

Each tool in `TOOLS` is a pure Python function over already-computed
Phase 5/6/7 outputs — no embeddings, no re-scoring, no re-filtering beyond
simple column lookups. Adding a new tool means: (1) write the function,
(2) add it to `TOOL_FUNCTIONS`, (3) add a matching JSON-schema entry to
`TOOLS`. The assertion in §4 fails loudly if the two ever drift apart.

## Known limitations
- The agent cannot change candidate preferences or re-run Phases 2-6;
  it only explores the results already computed for this candidate_id.
- `browse_jobs` requires `matched_jobs.parquet` or `reranked_jobs.parquet`
  to have been uploaded — without them it degrades gracefully with a
  clear error rather than failing silently.
- Grounding check (`check_grounding`) is a heuristic regex scan for
  `job_...`-shaped tokens, not a full fact-verification system — it
  catches obviously fabricated IDs, not subtler hallucinations (e.g. a
  real job_id with an invented skill attached to it).
"""

(OUT / "PHASE8_DATA_DICTIONARY.md").write_text(DATA_DICTIONARY, encoding="utf-8")
print("Data dictionary written.")

---
## 10. Persist outputs

In [ ]:
zip_path = shutil.make_archive("phase8_outputs", "zip", OUT.parent)
print(f"Created {zip_path} ({Path(zip_path).stat().st_size/1e6:.2f} MB)")

_colab_files.download(zip_path)

---
## الخطوة التالية — PHASE 9

`agent_conversation_log.json` + مخرجات Phase 5/6/7 تشكّل مدخلًا مباشرًا
لتقييم شبه رقمي: لكل سؤال تجريبي، تحقق هل الأداة الصحيحة اسُتدعيت، وهل
`unknown_ids_flagged` بقيت فارغة عبر عيّنة أسئلة (مؤشر أساسي على الاتساق).
التقييم الكمي الأساسي (Precision/Recall/NDCG) يبقى على مخرجات Phase 5/6
نفسها، مقارنة بمجموعة اختبار مُعلَّمة يدويًا كما خُطط له سابقًا في Phase 2.